# Lab 1 — Feed-Forward Neural Network (FFN)

In this lab you'll build your first neural network: a **feed-forward network**
(also called a multi-layer perceptron) that classifies clothing images from
**Fashion-MNIST** into 10 categories.

Along the way you will:

- Load a built-in dataset the easy way, **and** write your own `Dataset` class
- Build a simple FFN with `nn.Module`
- Train, validate, and evaluate it
- Measure accuracy, precision, recall, and a confusion matrix
- Plot loss and accuracy curves
- Peek inside the network by visualizing **layer activations**

Fashion-MNIST images are 28x28 grayscale. An FFN can't "see" 2-D structure, so
we flatten each image into a vector of 784 numbers and feed it through a few
fully-connected layers.

## 1. Setup and data

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

In [ ]:
torch.manual_seed(0)
np.random.seed(0)

# Reuse the Fashion-MNIST already stored in the project's data/ folder if we can
# find it (the notebooks live in sub-folders), otherwise download into ./data.
_candidates = ["./data", "../data", "../../data"]
DATA_ROOT = next(
    (p for p in _candidates if os.path.isdir(os.path.join(p, "FashionMNIST"))),
    "../data",
)
print("Data folder:", DATA_ROOT)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
DATA_ROOT

In [ ]:
# Fashion-MNIST has 10 clothing classes.
CLASS_NAMES = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]
NUM_CLASSES = len(CLASS_NAMES)

### Load Fashion-MNIST (the easy way)

In [ ]:
# The "simple" way: torchvision gives us a ready-made Dataset object.
# transforms.ToTensor() converts a PIL image (0-255) to a float tensor in [0, 1]
# and adds the channel dimension -> shape [1, 28, 28].
transform = transforms.ToTensor()

train_full = datasets.FashionMNIST(
    root=DATA_ROOT, train=True, download=True, transform=transform
)
test_set = datasets.FashionMNIST(
    root=DATA_ROOT, train=False, download=True, transform=transform
)

print("Training examples:", len(train_full))
print("Test examples:    ", len(test_set))

img, label = train_full[0]
print("One image tensor shape:", img.shape, "| label:", label, "=", CLASS_NAMES[label])

In [ ]:
train_full[0][1]

In [ ]:
# Let's look at a few images so we know what we're classifying.
fig, axes = plt.subplots(2, 5, figsize=(9, 4))
for ax, (img, label) in zip(axes.ravel(), train_full):
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(CLASS_NAMES[label], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

### Load the data yourself: a custom `Dataset`

The easy way is convenient, but you should know how to load your own data. A
custom `Dataset` gives you full control. We'll use this one for the rest of the
lab.

In [ ]:
# TODO: complete the custom Dataset class.
# A Dataset needs three methods: __init__, __len__, __getitem__.


class FashionCustomDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        # TODO: return the number of samples
        pass

    def __getitem__(self, idx):
        image = self.images[idx]  # [28, 28] uint8
        label = int(self.labels[idx])
        # TODO: turn `image` into a float tensor in [0, 1] with shape [1, 28, 28].
        #   hint: torch.from_numpy(...).float().div(255.0).unsqueeze(0)
        # TODO: apply self.transform if it is not None
        # TODO: return (image, label)
        pass


train_images = train_full.data.numpy()
train_labels = train_full.targets.numpy()
test_images = test_set.data.numpy()
test_labels = test_set.targets.numpy()

custom_train = FashionCustomDataset(train_images, train_labels)
custom_test = FashionCustomDataset(test_images, test_labels)

img_c, label_c = custom_train[0]
print("Custom sample shape:", img_c.shape, "| label:", CLASS_NAMES[label_c])

### DataLoaders (train / validation / test)

In [ ]:
# We split the training data into train/validation and wrap everything in
# DataLoaders. A DataLoader batches the data and shuffles it each epoch.
BATCH_SIZE = 128

val_size = 10_000
train_size = len(custom_train) - val_size
train_ds, val_ds = random_split(
    custom_train, [train_size, val_size], generator=torch.Generator().manual_seed(0)
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(custom_test, batch_size=BATCH_SIZE, shuffle=False)

print(
    f"train batches: {len(train_loader)}, "
    f"val batches: {len(val_loader)}, test batches: {len(test_loader)}"
)

# Every batch looks the same no matter which model we build:
xb, yb = next(iter(train_loader))
print("batch images:", xb.shape, "| batch labels:", yb.shape)

In [ ]:
def model_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 2. Build the model

In [ ]:
# TODO: build a feed-forward network.
# Suggested architecture:
#   Flatten  -> Linear(784, 256) -> ReLU
#            -> Linear(256, 128) -> ReLU
#            -> Linear(128, num_classes)
# Do NOT apply softmax at the end; CrossEntropyLoss handles that for us.
class FeedForwardNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.flatten = nn.Flatten()
        # TODO: define self.fc1, self.fc2, self.fc3

    def forward(self, x):
        x = self.flatten(x)
        # TODO: pass x through the layers with F.relu between them
        # TODO: return the final logits
        pass


model = FeedForwardNet(NUM_CLASSES)
print(model)
print("Number of trainable parameters:", model_parameters(model))

## 3. Train and validate

These helper functions are reused for every model in the course.

In [ ]:
# These three helpers work for EVERY model in this lab, because each model's
# forward() method reshapes the [batch, 1, 28, 28] input as it needs.


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()  # clear old gradients
        outputs = model(images)  # forward pass -> logits [batch, 10]
        loss = criterion(outputs, labels)
        loss.backward()  # backprop
        optimizer.step()  # update weights

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()  # no gradients needed for evaluation
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

    avg_loss = running_loss / total
    accuracy = correct / total
    preds = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    return avg_loss, accuracy, preds, labels

In [ ]:
def fit(model, epochs=5, lr=1e-3):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        va_loss, va_acc, _, _ = evaluate(model, val_loader, criterion)
        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        print(
            f"Epoch {epoch:2d}/{epochs} | "
            f"train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
            f"val loss {va_loss:.3f} acc {va_acc:.3f}"
        )
    return history

In [ ]:
# TODO: create the model and train it by calling fit(...).
#   model = FeedForwardNet(NUM_CLASSES)
#   history = fit(model, epochs=5, lr=1e-3)
model = FeedForwardNet(NUM_CLASSES)
history = fit(model, epochs=5, lr=1e-3)

## 4. Loss and accuracy curves

In [ ]:
def plot_curves(history):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

    ax1.plot(epochs, history["train_loss"], "o-", label="train")
    ax1.plot(epochs, history["val_loss"], "o-", label="val")
    ax1.set_title("Loss")
    ax1.set_xlabel("epoch")
    ax1.legend()

    ax2.plot(epochs, history["train_acc"], "o-", label="train")
    ax2.plot(epochs, history["val_acc"], "o-", label="val")
    ax2.set_title("Accuracy")
    ax2.set_xlabel("epoch")
    ax2.legend()

    plt.tight_layout()
    plt.show()


plot_curves(history)

## 5. Evaluate: accuracy, precision, recall

In [ ]:
# Run the model on the held-out test set, then compute standard metrics.
criterion = nn.CrossEntropyLoss()
test_loss, test_acc, preds, labels = evaluate(model, test_loader, criterion)

# "macro" averaging treats every class equally (good for balanced data).
precision = precision_score(labels, preds, average="macro", zero_division=0)
recall = recall_score(labels, preds, average="macro", zero_division=0)

print(f"Test loss     : {test_loss:.3f}")
print(f"Test accuracy : {test_acc:.3f}")
print(f"Precision(macro): {precision:.3f}")
print(f"Recall(macro)   : {recall:.3f}")

### Confusion matrix

In [ ]:
# A confusion matrix shows which classes get mixed up with which.
cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
ax.set_title("Confusion matrix (test set)")
plt.tight_layout()
plt.show()

## Visualizing layer activations

A network transforms an image step by step. We can attach **forward hooks** to
each layer to capture its output ("activation") for a sample image, then look at
how "lit up" each layer is. Bright cells = neurons responding strongly.

In [ ]:
# Forward hooks let us grab the output of any layer without changing the model.
activations = {}


def save_activation(name):
    def hook(module, inp, out):
        activations[name] = out.detach().cpu()

    return hook


# Register a hook on the two hidden layers.
h1 = model.fc1.register_forward_hook(save_activation("fc1 (256)"))
h2 = model.fc2.register_forward_hook(save_activation("fc2 (128)"))

# Push a single test image through the model.
model.eval()
sample_img, sample_label = custom_test[0]
with torch.no_grad():
    _ = model(sample_img.unsqueeze(0).to(device))

h1.remove()
h2.remove()  # clean up the hooks

# Show the image, then the activation of each hidden layer as a bar of neurons.
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
axes[0].imshow(sample_img.squeeze(), cmap="gray")
axes[0].set_title(f"input: {CLASS_NAMES[sample_label]}")
axes[0].axis("off")

for ax, (name, act) in zip(axes[1:], activations.items()):
    vec = act[0]  # activations for this one image
    ax.imshow(vec.reshape(1, -1), aspect="auto", cmap="viridis")
    ax.set_title(f"{name} activations")
    ax.set_yticks([])
    ax.set_xlabel("neuron")
plt.tight_layout()
plt.show()

for name, act in activations.items():
    print(
        f"{name}: mean={act.mean():.3f}, "
        f"fraction active (>0)={(act > 0).float().mean():.2f}"
    )

## 6. Your turn

Try these small experiments to build intuition:

1. Add another hidden layer, or make the layers wider. Does accuracy improve?
2. Change the learning rate to `1e-2` and `1e-4`. What happens to the curves?
3. Train for more epochs. When does validation accuracy stop improving?
4. Which two classes are most often confused? (Look at the confusion matrix.)